# Packages + Data load

In [1]:
import kagglehub
from dotenv import load_dotenv
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.special import inv_boxcox

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV

from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.cross_decomposition import PLSRegression

c:\Users\RobertMorsch\Practice_Projects\Ames-House-Prices\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os

# 1. Load variables from .env into the environment
load_dotenv() 

# 2. Access them using the standard OS library
kaggle_user = os.getenv('KAGGLE_USERNAME')
kaggle_key = os.getenv('KAGGLE_KEY')

# Download the latest version of the competition data
path = kagglehub.competition_download("house-prices-advanced-regression-techniques")

print("Path to competition files:", path)

for dirname, _, filenames in os.walk(path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

df = pd.read_csv(os.path.join(path, "train.csv"))
df_test = pd.read_csv(os.path.join(path, "test.csv"))

Path to competition files: C:\Users\RobertMorsch\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques
C:\Users\RobertMorsch\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques\data_description.txt
C:\Users\RobertMorsch\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques\sample_submission.csv
C:\Users\RobertMorsch\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques\test.csv
C:\Users\RobertMorsch\.cache\kagglehub\competitions\house-prices-advanced-regression-techniques\train.csv


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
df.info()
df.head()

Number of rows: 1460
Number of columns: 81
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
series = df.isna().sum().sort_values(ascending=False)
series[series > 0]

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
dtype: int64

In [5]:
cat_cols=df.select_dtypes(include=['object']).columns
num_cols = df.select_dtypes(include=['int64']).columns
print("Categorical Variables:")
print(len(cat_cols))
print(cat_cols.values)
print("\n")
print("Numerical Variables:")
print(len(num_cols))
print(num_cols.values)

Categorical Variables:
43
<StringArray>
[     'MSZoning',        'Street',         'Alley',      'LotShape',
   'LandContour',     'Utilities',     'LotConfig',     'LandSlope',
  'Neighborhood',    'Condition1',    'Condition2',      'BldgType',
    'HouseStyle',     'RoofStyle',      'RoofMatl',   'Exterior1st',
   'Exterior2nd',    'MasVnrType',     'ExterQual',     'ExterCond',
    'Foundation',      'BsmtQual',      'BsmtCond',  'BsmtExposure',
  'BsmtFinType1',  'BsmtFinType2',       'Heating',     'HeatingQC',
    'CentralAir',    'Electrical',   'KitchenQual',    'Functional',
   'FireplaceQu',    'GarageType',  'GarageFinish',    'GarageQual',
    'GarageCond',    'PavedDrive',        'PoolQC',         'Fence',
   'MiscFeature',      'SaleType', 'SaleCondition']
Length: 43, dtype: str


Numerical Variables:
35
<StringArray>
[           'Id',    'MSSubClass',       'LotArea',   'OverallQual',
   'OverallCond',     'YearBuilt',  'YearRemodAdd',    'BsmtFinSF1',
    'BsmtFinSF2',

C:\Users\RobertMorsch\AppData\Local\Temp\ipykernel_18704\584291390.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols=df.select_dtypes(include=['object']).columns


In [6]:
df_ID = df['Id']
df.drop('Id', axis=1, inplace=True)
df['MSSubClass'] = df['MSSubClass'].astype(str)

df['YearBuilt_AGE'] = df['YrSold'] - df['YearBuilt']
df['YearReMod_AGE'] = df['YrSold'] - df['YearRemodAdd']
df['GarageYrBlt_AGE'] = df['YrSold'] - df['GarageYrBlt']

df['WasReModded'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

In [7]:
# Dummy fields for nominal categorical columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

In [8]:
df_encoded.shape

(1460, 249)

In [9]:
def convert_to_dollars(y_preds, lam = None, run=0):
    if run == 0:
        return y_preds
    else:
        return inv_boxcox(y_preds, lam)

In [10]:
y_original = df['SalePrice']
Y = pd.Series(df['SalePrice'])
X = df_encoded.drop('SalePrice', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

_, _, y_train_original, y_test_original = train_test_split(X, y_original, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)
y_preds = ols_model.predict(X_test_scaled)

r_squared_score = ols_model.score(X_test_scaled, y_test)

# Inverse transform back to dollar scale
ols_preds_dollars = convert_to_dollars(y_preds, lam, 1)
mse = mean_squared_error(y_test_original, ols_preds_dollars)
print(f"Mean Squared Error: {mse}")
ols_rmse = np.sqrt(mse)
print(f"Root Mean Squared Error: {ols_rmse}")
print(f"R-squared score: {r_squared_score}")
print(f"Intercept: {ols_model.intercept_}")

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values